# 03 — Evaluation Harness

## Why this matters

Building the scorecard before any model exists is deliberate, not just tidy sequencing: if
the metric is designed *after* seeing which model looks good, it's easy to unconsciously pick
the metric that flatters that model. The harness here — the split, the candidate universe, the
ground truth, the metrics — is fixed first and reused unchanged by every model in notebooks
4-7, so every comparison in notebook 7's results table is apples-to-apples.

In [3]:
import numpy as np

from groovn.data import build_interaction_matrix, filter_min_interactions, load_ratings
from groovn.evaluation import build_ground_truth, warm_user_rows
from groovn.metrics import evaluate_rankings
from groovn.split import temporal_split

RANDOM_STATE = 42
ratings = load_ratings()
len(ratings)

4772071

## k-core filtering

Notebook 1 found 63% of users and 45% of items have exactly one rating. A model (kNN, ALS,
or the two-tower) has no signal to learn a 1-interaction user's taste from, and no signal to
place a 1-interaction item near anything else — keeping them in just inflates the matrix and
dilutes every similarity computation with noise. `filter_min_interactions` removes users and
items below an interaction floor, repeating until stable (needed because removing a user can
push one of *their* items below the floor too, and vice versa).

In [4]:
filtered = filter_min_interactions(ratings, min_user=5, min_item=5)
print(f"before: {len(ratings):,} rows, {ratings['user_id'].nunique():,} users, {ratings['item_id'].nunique():,} items")
print(f"after:  {len(filtered):,} rows, {filtered['user_id'].nunique():,} users, {filtered['item_id'].nunique():,} items")
print(f"kept {len(filtered) / len(ratings):.1%} of rows")

before: 4,772,071 rows, 1,754,118 users, 701,673 items
after:  1,552,764 rows, 123,876 users, 89,370 items
kept 32.5% of rows


Dropping 92% of *users* only costs 26% of *rows* — exactly what the Lorenz curve in notebook
1 predicted: the long tail of one-off raters contributes very little volume. What's left is a
matrix dense enough for kNN and MF to actually find structure in.

## Temporal split

Per notebook 2's leakage discussion: one global timestamp cutoff, everything before is train,
everything after is test. 80/20 by time, not by row count — a quantile on `ts` rather than a
fixed calendar date, so this still works if the dataset is refreshed with more recent years.

In [6]:
train, test, cutoff = temporal_split(filtered, test_frac=0.2)
print(f"cutoff: {cutoff}")
print(f"train: {len(train):,} rows, test: {len(test):,} rows")

cutoff: 2017-07-20 01:40:01.216000
train: 1,242,211 rows, test: 310,553 rows


## Candidate universe, warm users, and ground truth

Three definitions that every model's evaluation run depends on, so they're centralized in
`groovn.evaluation` rather than redefined per notebook:

- **Candidate universe**: only items present in `train`. A model that never saw an item during
  training has no factors/neighbors for it and can't meaningfully score it.
- **Warm test users**: test users who also appear in `train`. A true cold-start user (first
  interaction ever falls after the cutoff) has no history for any of these models to use — a
  real limitation, not a bug, and one we report rather than hide (notebook 7 covers it as its
  own segment).
- **Ground truth**: for each warm test user, the set of train-catalog items they interact
  with *after* the cutoff — what a good ranking should have surfaced.

In [7]:
matrix, user_index, item_index = build_interaction_matrix(train, value_col="rating")
warm_rows = warm_user_rows(test, user_index)
ground_truth = build_ground_truth(test, user_index, item_index)

print(f"train matrix: {matrix.shape[0]:,} users x {matrix.shape[1]:,} items, {matrix.nnz:,} interactions")
print(f"warm test users: {len(warm_rows):,} / {test['user_id'].nunique():,} total test users")
print(f"warm users with >=1 train-catalog item in their test interactions: {len(ground_truth):,}")

train matrix: 111,261 users x 82,723 items, 1,242,211 interactions
warm test users: 39,843 / 52,458 total test users
warm users with >=1 train-catalog item in their test interactions: 32,384


## Metrics: Recall@K, NDCG@K, coverage@K

- **Recall@K**: of the items a user actually interacted with later, what fraction did the
  top-K ranking catch? `0` to `1`, higher is better.
- **NDCG@K**: like recall, but a hit ranked 1st counts more than one ranked 10th (discounted
  by `log2(rank+1)`), normalized against the best possible ordering so scores are comparable
  across users with different numbers of relevant items.
- **Coverage@K**: what fraction of the *entire catalog* ever shows up in anyone's top-K? A
  model that recommends the same 20 popular albums to everyone can still score well on
  recall/NDCG (popular albums get interacted with a lot) while being useless for catalog
  discovery — coverage is the metric that catches that.

Implementations are in `groovn.metrics`, already unit-tested on synthetic cases in
`tests/test_metrics.py`. What's missing is a check that they behave sanely on the *real*
candidate universe and ground truth above — the actual point of a harness built before any
model exists.

## Sanity check: random floor and oracle ceiling

Before trusting these metrics for a real model, run them against two trivial rankers that
have known correct answers:
- **Random**: `k` uniformly random items per user. Should score close to `0` on recall/NDCG
  (hitting a specific relevant item by chance, out of tens of thousands of candidates, is
  rare) and close to `1` on coverage (random draws touch almost everything eventually).
- **Oracle**: literally hands back each user's own ground-truth items. Should score close to
  the maximum possible on recall/NDCG — not exactly `1.0` for recall, since a user with more
  than `k` relevant items can't have all of them in a top-`k` list by definition.

If the harness disagreed with either of these, the bug would be in the harness, not in a
future model — far easier to catch now than after training something expensive.

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
n_items = matrix.shape[1]
K = 10

random_rankings = {u: rng.choice(n_items, size=K, replace=False).tolist() for u in warm_rows}
oracle_rankings = {u: list(ground_truth.get(u, []))[:K] for u in warm_rows}

print("random: ", evaluate_rankings(random_rankings, ground_truth, k=K, catalog_size=n_items))
print("oracle: ", evaluate_rankings(oracle_rankings, ground_truth, k=K, catalog_size=n_items))

random:  {'recall@10': 0.0001724059309625715, 'ndcg@10': 0.00011474566413089176, 'coverage@10': 0.9919853003396879, 'n_users_evaluated': 32384}
oracle:  {'recall@10': 0.9729335204142625, 'ndcg@10': 1.0, 'coverage@10': 0.478681865986485, 'n_users_evaluated': 32384}


**Reading the result:** random lands at recall@10 ≈ 0.0002 and NDCG@10 ≈ 0.0001 — essentially
zero, as expected — while covering ~99% of the catalog (anything random eventually samples
almost everything). Oracle lands at recall@10 ≈ 0.97 (not 1.0 — some users have more than 10
relevant items after the cutoff, so a top-10 list can't contain all of them) and NDCG@10 is
exactly 1.0 by construction (every slot in the oracle's top-10 is a true hit, which is the
best possible ordering). Oracle's coverage@10 is only ~0.48, lower than random's — it only
ever recommends items people actually went on to interact with, a smaller and more
popularity-skewed slice of the catalog than a uniform random draw touches. That gap is the
whole reason to track coverage *alongside* recall/NDCG rather than instead of it: a model
could chase recall by leaning on popular items and quietly lose coverage, and this harness
would catch it.

Both extremes behave exactly as expected — the harness is trustworthy to hand to a real model.

## Handing off

Notebooks 4-7 all start the same way: `filter_min_interactions` -> `temporal_split` ->
`build_interaction_matrix` -> `warm_user_rows` / `build_ground_truth` -> `evaluate_rankings`
at `K=10`. That's `groovn.data.load_filtered_split()` plus the two `groovn.evaluation` calls
above — a model only needs to produce a `scores` object (vector, sparse matrix, or callable)
that `groovn.ranking.top_k_excluding_seen` can turn into rankings.